In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

import sys
import os
import pickle
from tqdm import tqdm
# Get the absolute path to the folder containing `utils`
utils_path = os.path.abspath('../')
if utils_path not in sys.path:
    sys.path.append(utils_path)

os.environ["CUDA_VISIBLE_DEVICES"] = "2" # second gpu
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
    
# from utils import *

# az.style.use("arviz-docgrid")
plt.rcParams['figure.dpi'] = 140

experiment_orientations = [159, 123, 87, 51, 15]
subjects = ["01", "02", "03", "04", "05", "06", "07" ,"09", "10", "11", "12"]
median_key = {15:0, 51:1, 87:2, 123:3, 159:4}
std_key = {15:0, 51:1, 87:2, 123:3, 159:4}

c_table = pd.read_csv('../data_caches/ctiraltable.csv')
med = np.load('../data_caches/med.npy')
std = np.load('../data_caches/std.npy')
ctimetable = np.load('../data_caches/ctimetable.npy')
r_table = pd.read_csv('../data_caches/rtrialtable.csv', index_col=0)
(x, y, d, r, e, cd, ce) = np.load('../data_caches/rtimetable.npy', allow_pickle=True)

In [14]:
xaxis = np.arange(-250, 750, 1) * (1000/120)
start_idx, end_idx = np.searchsorted(xaxis, -500), np.searchsorted(xaxis, 1500)
(start_idx, end_idx)

# our frmes of intrest are only
er = e[:, :, :, :, start_idx:end_idx].reshape(-1, 239)

# ds
ds = cd[:, :, :, :, start_idx:end_idx].reshape(-1, 239)

er[np.isnan(er)] = 90
ds[np.isnan(ds)] = 0

# min max scale err
er = er / 180

emissions = np.array(np.stack([ds, er], axis=-1))

subset_idx = np.random.choice(np.arange(0, len(emissions)), 400, replace=False)
sub_em = emissions[subset_idx]

er.shape, np.isnan(er).any(), ds.shape, np.isnan(ds).any(), emissions.shape

((34560, 239), False, (34560, 239), False, (34560, 239, 2))

In [15]:
from sklearn.mixture import GaussianMixture

In [16]:
# Fit a single-component Gaussian on training data:
gmm = GaussianMixture(n_components=1, covariance_type='full', random_state=0)
gmm.fit(train_data)

# Compute total log likelihood on held-out data.
# Note: gmm.score returns the average log likelihood per sample.
ll_held = gmm.score(held_out_data) * held_out_data.shape[0]

# Compute AIC and BIC on training data:
aic = gmm.aic(train_data)
bic = gmm.bic(train_data)

print("Log-likelihood on held-out data (scikit-learn):", ll_held)
print("AIC (scikit-learn):", aic)
print("BIC (scikit-learn):", bic)

NameError: name 'train_data' is not defined